# Activation Steering — 모듈 3(추출) + 4a(진단), Gemma 2 2B @ Colab

외향성 대조쌍(`pairs_final.jsonl`, 1,200쌍)으로 **CAA 벡터 추출**과 **V1 vs V2 진단**을 GPU에서 실행한다.

**실행 순서**: GPU 확인 → 의존성 설치 → HF 로그인 → 번들 업로드 → 스모크 → 전체 추출 → 4a 진단 → 산출물 다운로드.

> ⚠️ **사전 준비**: [huggingface.co/google/gemma-2-2b](https://huggingface.co/google/gemma-2-2b) 에서 **Gemma 라이선스 수락** 후, HF **Access Token**(read) 발급. (게이트 모델)
> 런타임 → 런타임 유형 변경 → **GPU(T4)** 선택.


## 1. GPU 확인


In [ ]:
!nvidia-smi

## 2. 의존성 설치
Colab 엔 torch/numpy/tqdm 기본 탑재. transformers(Gemma2 지원)·accelerate와, 코드가 재사용하는 openai/dotenv(상수용)만 추가.


In [ ]:
!pip install -q "transformers>=4.42" accelerate huggingface_hub openai python-dotenv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Hugging Face 로그인 (Gemma 게이트 접근)
발급한 read 토큰을 입력. 이 세션에서 `google/gemma-2-2b` 다운로드에 사용된다.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. 코드+데이터 번들 업로드
로컬에서 만든 **`colab_bundle.zip`** 을 선택해 업로드한다. (common.py + steering/*.py + data/facets_en.json + outputs/pairs_final.jsonl)


In [ ]:
from google.colab import files
import zipfile, os
print('colab_bundle.zip 를 선택하세요 ...')
up = files.upload()
name = next(iter(up))
os.makedirs('/content/asteer', exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall('/content/asteer')
%cd /content/asteer
print('\n구조:'); import subprocess; print(subprocess.run(['find','.','-maxdepth','2','-type','f'],capture_output=True,text=True).stdout)

## 5. 스모크 (문장 8개) — 파이프라인 정상 확인
`pooled_mean/last.npy` shape 와 `hidden_states` 정합(27×2304)을 빠르게 검증.


In [ ]:
!python steering/extract_activations.py --limit 8

## 6. 전체 추출 (1,200쌍 × 2 = 2,400 forward)
레이어별 pooled 활성화만 캐시 → `artifacts/activations/pooled_{mean,last}.npy [2400,27,2304] fp16` (~0.6GB). T4로 수분.


In [ ]:
!python steering/extract_activations.py

## 7. 4a 진단 — V1 vs V2
예시별 L2→facet 균등(V1) vs 전체 풀링(V2) 비교. cos(V1,V2)·facet 기여도·6×6 코사인·PC1·leave-one-facet-out.
핵심: **개수 균등이어도 norm 차이로 실질 기여가 불균등**한지 확인 → 채택 변형·레이어 결정.


In [ ]:
!python steering/diagnostics.py

## 8. 결과 요약 + 산출물 다운로드


In [ ]:
import json
d = json.load(open('artifacts/vectors/diagnostics.json'))
print('layer | cos(V1,V2) | 최대기여 facet(비율) | PC1(설명분산)')
for L, r in d['layers'].items():
    mf = r['max_contrib_facet']
    print(f"  {L:>2}   |   {r['cos_V1_V2']:.3f}    | {mf} ({r['facet_contrib_to_V2'][mf]:.2f})        | {r['svd']['pc1_frac']:.2f}")
print('\n해석: cos(V1,V2)>0.95 → 구성 무관(V1 채택). <0.9 → 4b 에서 V1·V2 둘 다 평가.')
# 진단 리포트(소용량) 다운로드
files.download('artifacts/vectors/diagnostics.json')

## 9. (선택) 전체 활성화 캐시를 Google Drive 로 보존
`pooled_*.npy` 는 크지만(~0.6GB), 저장해두면 로컬/재접속에서 **GPU 없이** V1/V2·레이어 스윕을 반복할 수 있다.


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, os
# os.makedirs('/content/drive/MyDrive/asteer_artifacts', exist_ok=True)
# shutil.copytree('artifacts', '/content/drive/MyDrive/asteer_artifacts', dirs_exist_ok=True)
# print('Drive 저장 완료')

---
**다음 단계**: 이 추출·진단이 검증되면 **4b(스티어링 α-sweep·Mini-IPIP)** 와 **5(Gemma Scope SAE)** 코드를 이 결과에 맞춰 작성한다. (지금 blind 작성은 위험)
